# Fine-tune `facebook/mms-tts-pcm` on the Honest Orb voice

Base recipe: [ylacombe/finetune-hf-vits](https://github.com/ylacombe/finetune-hf-vits) (`run_vits_finetuning.py` + JSON config, launched with `accelerate`).
Pidgin uses Latin script, so no `uroman` step is needed.

**Budget:** ~30–55 min training on a free Colab T4 (180 clips ≈ 11 steps/epoch × 100 epochs ≈ 1100 steps at ≈1.5–3 s/step, fp16), plus ~10 min setup. Total well under 1.5 h.

Before you run:

1. Colab menu → Runtime → **Change runtime type → T4 GPU**.
2. Zip the prepared dataset from the repo: `cd data/voice && zip -r voice_dataset.zip dataset` (made by `python api/voice_prep.py`).
3. Create a Hugging Face token with **write** access and save it as a Colab secret named `HF_TOKEN` (or log in when asked).

In [ ]:
# 0. Confirm we have a GPU (want to see a Tesla T4)
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
# 1. Recipe + deps + the Cython monotonic-alignment op (required: the pure-Python one is far too slow)
!git clone https://github.com/ylacombe/finetune-hf-vits.git
%cd /content/finetune-hf-vits
!pip -q install -r requirements.txt datasets accelerate
!cd monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace && cd ..

In [ ]:
# 2. Hugging Face login (pushes the dataset + the finetuned model to your account)
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=token)
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()  # paste a token with write access

HANDLE = input('Your HF username (e.g. octopus): ').strip()  # used for repo names below

## 3. Upload the dataset

Upload the `voice_dataset.zip` you made from `data/voice/dataset` (contains `wavs/line_*.wav` + `metadata.csv` with `file|text` rows).

In [ ]:
from google.colab import files
up = files.upload()  # select voice_dataset.zip
!unzip -q voice_dataset.zip -d /content/voice_dataset
!head -3 /content/voice_dataset/metadata.csv && ls /content/voice_dataset/wavs | wc -l

In [ ]:
# 4. Wrap wavs + transcripts as a Hub dataset (train/test split; test doubles as the eval split)
import pandas as pd
from datasets import Dataset, Audio

df = pd.read_csv('/content/voice_dataset/metadata.csv', sep='|')  # columns: file, text
df['audio'] = df['file'].apply(lambda f: f'/content/voice_dataset/{f}')
ds = Dataset.from_pandas(df[['audio', 'text']]).cast_column('audio', Audio(sampling_rate=16000))
splits = ds.train_test_split(test_size=9, seed=42)  # 171 train / 9 eval
print(splits)
splits.push_to_hub(f'{HANDLE}/honest-orb-pcm-voice', private=True)

## 5. Build the Pidgin training checkpoint

There is no ready-made `mms-tts-pcm-train` checkpoint (only eng/spa/kor/mar/tam/guj exist), so per the recipe's Option 2 we convert the discriminator of `facebook/mms-tts-pcm` once. Keep this checkpoint: re-running finetunes later skips this step.

In [ ]:
# '--push_to_hub' also saves it remotely; drop that flag to keep it local-only
!python convert_original_discriminator_checkpoint.py --language_code pcm \
  --pytorch_dump_folder_path /content/mms-tts-pcm-train \
  --push_to_hub {HANDLE}/mms-tts-pcm-train

## 6. Finetune (the only long cell: ~30–55 min on a T4)

In [ ]:
# Training config: same loss weights / optimizer as the recipe's finetune_english.json,
# sized for 171 clips (batch 16 -> ~11 steps/epoch, 100 epochs -> ~1100 steps, fp16 on).
import json

config = {
    'project_name': 'honest_orb_pcm',
    'push_to_hub': True,
    'hub_model_id': f'{HANDLE}/honest-orb-mms-pcm',
    'overwrite_output_dir': True,
    'output_dir': '/content/honest-orb-pcm',
    'dataset_name': f'{HANDLE}/honest-orb-pcm-voice',
    'audio_column_name': 'audio',
    'text_column_name': 'text',
    'train_split_name': 'train',
    'eval_split_name': 'test',
    'max_duration_in_seconds': 20,
    'min_duration_in_seconds': 0.5,
    'max_tokens_length': 500,
    'model_name_or_path': f'{HANDLE}/mms-tts-pcm-train',
    'preprocessing_num_workers': 2,
    'do_train': True,
    'num_train_epochs': 100,
    'gradient_accumulation_steps': 1,
    'gradient_checkpointing': False,
    'per_device_train_batch_size': 16,
    'learning_rate': 2e-5,
    'adam_beta1': 0.8,
    'adam_beta2': 0.99,
    'warmup_ratio': 0.01,
    'group_by_length': False,
    'do_eval': True,
    'eval_steps': 100,
    'per_device_eval_batch_size': 8,
    'max_eval_samples': 9,
    'do_step_schedule_per_epoch': True,
    'weight_disc': 3,
    'weight_fmaps': 1,
    'weight_gen': 1,
    'weight_kl': 1.5,
    'weight_duration': 1,
    'weight_mel': 35,
    'fp16': True,
    'seed': 42,
}
with open('/content/honest_orb_pcm.json', 'w') as f:
    json.dump(config, f, indent=4)
print('wrote /content/honest_orb_pcm.json')

In [ ]:
!accelerate launch run_vits_finetuning.py /content/honest_orb_pcm.json

## 7. Test it and take it home

`push_to_hub: true` already uploaded the model to `{HANDLE}/honest-orb-mms-pcm`. The zip below is the local copy: unzip it next to the repo and point `api/tts.py`'s `MODEL_ID` at that folder (it loads via `AutoTokenizer`/`VitsModel`, same as today).

In [ ]:
# Quick listen: Pidgin line in the new voice
from transformers import pipeline
import scipy.io.wavfile

synth = pipeline('text-to-speech', f'{HANDLE}/honest-orb-mms-pcm')
out = synth('Oga officer no fit search your phone without reason, stay calm.')
scipy.io.wavfile.write('/content/honest_orb_test.wav', rate=out['sampling_rate'], data=out['audio'])
from IPython.display import Audio
Audio('/content/honest_orb_test.wav')

# Local copy for the repo machine
!zip -qr /content/honest_orb_pcm_finetuned.zip /content/honest-orb-pcm
from google.colab import files
files.download('/content/honest_orb_pcm_finetuned.zip')

## Notes

- Nothing is trained until you run section 6; this notebook only prepared the recipe.
- License: `facebook/mms-tts-pcm` is CC BY-NC 4.0, and the finetuned voice inherits it — non-commercial use only, same as the base model.
- If training is interrupted, re-run from section 6 with `overwrite_output_dir: false` to resume from the last checkpoint in `/content/honest-orb-pcm`.